# Pipeline de modelagem e registro de experimentos (Etapa 3)
Projeto de previsão de demanda de energia elétrica para apoio ao planejamento de capacidade da Copel.

A pergunta de negócio é qual será a carga diária do subsistema Sul (MWmed) daqui a 7 dias. A Etapa 1 pediu previsões diárias para os próximos dias ou semanas, e uma semana de antecedência dá tempo de ajustar contratação, escala de equipes e manobras de rede.

Cada etapa do pipeline é uma função, e todas as execuções passam pelas mesmas funções.

| Etapa | O que faz | Onde está |
|---|---|---|
| 1. Dados brutos | Downloads com hash SHA-256 (ONS, ANEEL, EPE, Open-Meteo) e integração | notebooks das Etapas 2 e 3 (integração), aqui só lemos a versão registrada |
| 2. Validação | Confere esquema, datas, alvo e faixas e bloqueia variáveis que vazariam informação | seção 3 |
| 3. Preparação | Série diária contínua, sem dias imputados, com treino, validação e teste separados no tempo | seção 4 |
| 4. Variáveis | Calendário, histórico da carga com 7 dias de atraso, temperatura e perfil de consumo da EPE | seção 5 |
| 5. Treinamento | Regra simples (referência), Regressão Linear, Random Forest, Gradient Boosting e Prophet | seção 7 |
| 6. Avaliação | MAE, RMSE, MAPE, viés e erro nos dias de pico comparados à regra simples, e explicabilidade (XAI) do modelo escolhido | seções 7 e 8 |

Cada execução de modelo (run) é gravada em `registro_experimentos.csv` com a versão dos dados (hash), a versão do código do pipeline (hash), o algoritmo, os parâmetros, as variáveis, as janelas de tempo, as métricas e a data da execução.

Para rodar no Google Colab, use Ambiente de execução > Executar tudo. Se o notebook de integração não rodou na mesma sessão, o notebook pede o `etapa3_resultados.zip` (ou só o `dataset_integrado.csv`).

## 1. Configuração

In [ ]:
import hashlib, inspect, io, json, platform, shutil, time, zipfile
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.inspection import permutation_importance
import joblib
import logging
try:
    from prophet import Prophet
except ImportError:                      # instala fora do Colab
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "prophet"], check=True)
    from prophet import Prophet
for _nome in ("cmdstanpy", "prophet"):
    logging.getLogger(_nome).setLevel(logging.ERROR)

# PARÂMETROS GERAIS
HORIZONTE_DIAS = 7                      # previsão de D+7 com dados até D
SEMENTE = 42                            # semente fixa
JANELAS = {                             # divisão no tempo
    "treino":    ("2017-01-01", "2023-12-31"),
    "validacao": ("2024-01-01", "2024-12-31"),   # escolha do modelo
    "teste":     ("2025-01-01", "2025-12-31"),   # só confirmação
}
PERCENTIL_PICO = 0.90                   # 10% de dias com maior carga
TOLERANCIA_EMPATE = 0.01                # até 1% de MAE conta como empate

# PASTAS
BASE = Path("etapa3")
ENTRADA = BASE / "saidas"
MOD = BASE / "modelagem"; OUT = MOD / "saidas"; FIG = MOD / "figuras"; MODELOS = MOD / "modelos"
for p in (ENTRADA, OUT, FIG, MODELOS):
    p.mkdir(parents=True, exist_ok=True)
ARQ_DADOS = ENTRADA / "dataset_integrado.csv"
ARQ_DICIONARIO = ENTRADA / "dicionario_integrado.csv"
ARQ_REGISTRO = OUT / "registro_experimentos.csv"

DATA_EXECUCAO = datetime.now()
ID_EXECUCAO = DATA_EXECUCAO.strftime("%Y%m%d_%H%M%S")

plt.rcParams.update({
    "figure.figsize": (11, 4.2), "figure.dpi": 110, "savefig.dpi": 200,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.color": "#DDDDDD", "grid.linewidth": 0.6,
    "axes.titlesize": 12, "axes.titleweight": "bold", "font.size": 10,
})
COR_SERIE, COR_REF, COR_ALERTA, COR_2 = "#1F5F8B", "#7A7A7A", "#C0392B", "#D9822B"

import prophet as _prophet
AMBIENTE = {"python": platform.python_version(), "pandas": pd.__version__, "numpy": np.__version__,
            "scikit-learn": sklearn.__version__, "prophet": _prophet.__version__}
print("Execução:", ID_EXECUCAO, "(o Colab usa o horário UTC)")
print(AMBIENTE)

## 2. Dados brutos (versão de entrada)

Os dados brutos foram baixados e integrados pelo notebook de integração, que registrou cada download com hash. Aqui lemos o resultado e calculamos o hash do arquivo de entrada, que identifica qual versão dos dados cada modelo usou. Se alguém rodar de novo com outro arquivo, o hash muda e o registro mostra isso.

In [ ]:
if not ARQ_DADOS.exists():
    try:
        from google.colab import files
    except ImportError:
        raise FileNotFoundError(f"Coloque o dataset integrado em {ARQ_DADOS} (ou o zip do notebook de integração).")
    print("Envie o etapa3_resultados.zip (ou o dataset_integrado.csv) gerado pelo notebook de integração.")
    enviados = files.upload()
    for nome, conteudo in enviados.items():
        if nome.lower().endswith(".zip"):
            with zipfile.ZipFile(io.BytesIO(conteudo)) as z:
                for m in z.namelist():
                    if m.startswith("saidas/") and not m.endswith("/"):
                        (ENTRADA / Path(m).name).write_bytes(z.read(m))
        elif nome.lower().endswith(".csv"):
            destino = ARQ_DICIONARIO if "dicionario" in nome else ARQ_DADOS
            destino.write_bytes(conteudo)
    if not ARQ_DADOS.exists():
        raise FileNotFoundError("O arquivo enviado não contém o dataset_integrado.csv.")

def sha256(caminho):
    h = hashlib.sha256()
    with open(caminho, "rb") as f:
        for bloco in iter(lambda: f.read(1 << 20), b""):
            h.update(bloco)
    return h.hexdigest()

def carregar_dados_brutos(caminho):
    """Lê a versão integrada e identifica a versão."""
    df = pd.read_csv(caminho, parse_dates=["data"], low_memory=False)
    versao = {"arquivo": caminho.name, "sha256": sha256(caminho), "linhas": len(df), "colunas": df.shape[1],
              "inicio": df["data"].min().date().isoformat(), "fim": df["data"].max().date().isoformat()}
    params_int = ENTRADA / "parametros_execucao_etapa3.json"
    if params_int.exists():
        versao["integracao_executada_em"] = json.loads(params_int.read_text()).get("data_execucao")
    return df, versao

bruto, VERSAO_DADOS = carregar_dados_brutos(ARQ_DADOS)
ROTULO_DADOS = f"dataset_integrado@{VERSAO_DADOS['sha256'][:12]}"
print(json.dumps(VERSAO_DADOS, indent=2, ensure_ascii=False))

arq_dl = ENTRADA / "registro_downloads_etapa3.csv"
if arq_dl.exists():
    dl = pd.read_csv(arq_dl)
    print("\nLinhagem: arquivos brutos usados na integração (registro de downloads)")
    cols = [c for c in ("fonte", "arquivo", "sha256", "status") if c in dl]
    print(dl[cols].assign(sha256=dl["sha256"].astype(str).str[:12] if "sha256" in dl else None).to_string(index=False))

## 3. Validação de entrada

Antes de treinar, o pipeline confere o arquivo. Falhas graves interrompem a execução e as demais viram alerta no relatório.

A verificação mais importante para governança é a de vazamento de informação. Cada variável do modelo é procurada no dicionário de dados da integração, e colunas marcadas como análise, validação, sinalização ou que vazariam informação são bloqueadas. Um exemplo é o consumo da EPE do próprio mês, que só é publicado meses depois. As interrupções da ANEEL só podem escolher dias de treino e nunca entram como variável, porque no dia da previsão ainda não se sabe se haverá apagão.

In [ ]:
COLUNAS_OBRIGATORIAS = ["data", "carga_mwmed", "dia_semana", "mes", "feriado", "regime_metodologico",
                        "flag_imputado", "flag_evento_conhecido", "flag_grande_interrupcao",
                        "temp_media_sul", "temp_max_sul", "temp_min_sul", "graus_dia_calor", "graus_dia_frio"]
COLUNAS_SO_FILTRO = {"flag_grande_interrupcao", "flag_evento_conhecido", "flag_imputado"}   # só filtram dias
# colunas da Etapa 2 conhecidas antes do dia
CALENDARIO_ETAPA2 = {"data", "dia_semana", "mes", "feriado", "nome_feriado", "fim_de_semana", "regime_metodologico", "carga_mwmed"}

dicionario = pd.read_csv(ARQ_DICIONARIO) if ARQ_DICIONARIO.exists() else None
validacoes = []

def checar(nome, ok, detalhe, grave=False):
    validacoes.append({"verificacao": nome, "resultado": "OK" if ok else ("ERRO" if grave else "ALERTA"), "detalhe": detalhe})
    return ok

def validar_entrada(df):
    """Confere o arquivo e para em erro grave."""
    faltam = [c for c in COLUNAS_OBRIGATORIAS if c not in df]
    checar("Colunas obrigatórias presentes", not faltam, f"faltam {faltam}" if faltam else f"{len(COLUNAS_OBRIGATORIAS)} colunas", grave=True)
    dup = int(df["data"].duplicated().sum())
    checar("Uma linha por dia (sem datas repetidas)", dup == 0, f"{dup} datas repetidas", grave=True)
    esperado = pd.date_range(df["data"].min(), df["data"].max(), freq="D")
    lacunas = len(esperado.difference(df["data"]))
    checar("Calendário contínuo", lacunas == 0, f"{lacunas} dias ausentes no intervalo")
    alvo_nulo = int(df["carga_mwmed"].isna().sum())
    checar("Alvo preenchido", alvo_nulo == 0, f"{alvo_nulo} dias sem carga", grave=True)
    fora = int(((df["carga_mwmed"] < 3000) | (df["carga_mwmed"] > 30000)).sum())
    checar("Carga em faixa plausível (3 a 30 mil MWmed)", fora == 0, f"{fora} dias fora da faixa")
    t = df["temp_media_sul"]
    checar("Temperatura em faixa plausível (-5 a 40 °C)", bool(t.dropna().between(-5, 40).all()), f"mín {t.min():.1f}, máx {t.max():.1f}")
    for nome, (ini, fim) in JANELAS.items():
        n = int(df["data"].between(ini, fim).sum())
        dias = len(pd.date_range(ini, fim))
        checar(f"Janela de {nome} coberta pelos dados", n == dias, f"{n} de {dias} dias", grave=True)
    if dicionario is None:
        checar("Dicionário de dados disponível", False, "sem dicionário, a checagem de vazamento usa só a lista interna")
    else:
        checar("Dicionário de dados disponível", True, f"{len(dicionario)} campos descritos")
    return df

def uso_declarado(coluna):
    if dicionario is None or coluna not in set(dicionario["campo"]):
        return None
    return str(dicionario.loc[dicionario["campo"] == coluna, "uso_no_modelo"].iloc[0])

def validar_variaveis(origens):
    """Barreira contra vazamento pelo dicionário de dados."""
    bloqueadas = []
    for col in sorted(set(origens)):
        if col in COLUNAS_SO_FILTRO:
            bloqueadas.append(f"{col} (só pode escolher dias de treino)")
            continue
        uso = uso_declarado(col)
        if uso is None:
            if col not in CALENDARIO_ETAPA2:
                bloqueadas.append(f"{col} (não está no dicionário)")
            continue
        u = uso.lower()
        permitida = u.startswith("preditor") or (u.startswith("conforme etapa 2") and col in CALENDARIO_ETAPA2)
        if not permitida:
            bloqueadas.append(f"{col} (uso no dicionário: {uso})")
    if bloqueadas:
        raise ValueError("Variáveis bloqueadas pela validação: " + "; ".join(bloqueadas))
    return True

dados = validar_entrada(bruto)
tab_validacao = pd.DataFrame(validacoes)
if (tab_validacao["resultado"] == "ERRO").any():
    display(tab_validacao)
    raise RuntimeError("Validação de entrada falhou: corrija o arquivo antes de treinar.")
tab_validacao

Teste da barreira com o consumo da EPE do mês corrente, o número de interrupções do dia, a marcação de grande interrupção e a marcação de dia atípico (calculada com a própria carga do dia). As quatro devem ser recusadas. A temperatura está marcada como preditora no dicionário e deve passar.

In [ ]:
for tentativa in (["consumo_pr_total_mwh"], ["n_interrupcoes"], ["flag_grande_interrupcao"], ["flag_atipico"], ["temp_media_sul"]):
    try:
        validar_variaveis(tentativa)
        print(tentativa, "-> aceita")
    except ValueError as e:
        print(tentativa, "-> recusada:", str(e).split(": ", 1)[1])

## 4. Preparação

* A série fica em ordem de data, com um registro por dia.
* Dias com carga imputada na Etapa 2 não são observações reais. Ficam fora do treino e da avaliação, mas continuam como histórico.
* A divisão é no tempo. O modelo aprende com 2017 a 2023, é escolhido em 2024 e é confirmado em 2025. Misturar dias ao acaso deixaria o modelo ver o futuro, porque dias vizinhos são muito parecidos, e o erro sairia baixo demais.
* Os dados de antes de 2017 servem só para calcular o histórico, como a carga de um ano antes.
* Dias cujo histórico recente (até 21 dias antes) atravessa uma mudança de metodologia do ONS (mar/2021 e abr/2023) ficam fora do treino em todas as runs. Neles a semana anterior foi medida de outra forma e o modelo aprenderia um salto que não é do consumo.

In [ ]:
def preparar(df):
    """Ordena por dia e marca a janela de cada dia."""
    d = df.sort_values("data").set_index("data").asfreq("D")
    d["janela"] = "fora"
    for nome, (ini, fim) in JANELAS.items():
        d.loc[ini:fim, "janela"] = nome
    d["avaliavel"] = (d["flag_imputado"].fillna(0) == 0) & d["carga_mwmed"].notna()
    alcance = int(np.ceil(HORIZONTE_DIAS / 7) * 7) + 14          # dia mais antigo usado no histórico semanal
    regime = d["regime_metodologico"]
    d["historico_cruza_regime"] = (regime != regime.shift(alcance)).astype(int)
    return d

prep = preparar(dados)
resumo_janelas = (prep.groupby("janela")
                  .agg(dias=("carga_mwmed", "size"), dias_avaliaveis=("avaliavel", "sum"),
                       carga_media=("carga_mwmed", "mean"), grandes_interrupcoes=("flag_grande_interrupcao", "sum"),
                       eventos_conhecidos=("flag_evento_conhecido", "sum"),
                       transicao_metodologia=("historico_cruza_regime", "sum"))
                  .reindex(["treino", "validacao", "teste", "fora"]).round(0))
resumo_janelas

A carga média cresce de uma janela para a outra, pelo crescimento do consumo e pela mudança de metodologia do ONS em 2023. Modelos de árvore como o Random Forest não preveem valores acima dos que viram no treino. Por isso testamos também uma versão relativa, em que o modelo aprende a razão entre a carga do dia e a carga média da semana mais recente conhecida. Na versão relativa o nível já vem da semana de referência, então o indicador de regime metodológico deixa de fazer sentido. A run R04 mostra o que acontece quando ele é mantido.

## 5. Variáveis

Todas as variáveis usam só o que estaria disponível 7 dias antes do dia previsto.

| Grupo | Variáveis | Disponível na hora da previsão |
|---|---|---|
| Calendário | dia da semana, mês e dia do ano (em seno e cosseno), feriado, véspera e dia seguinte de feriado, fim de ano, regime metodológico do ONS e tendência (só na versão absoluta) | sim, conhecido com antecedência |
| Histórico | carga de 7, 14 e 21 dias antes (mesmo dia da semana), média dos 7 dias mais recentes conhecidos e carga de 364 dias antes | sim, só usa dias até D |
| Temperatura | média, máxima e mínima das capitais, graus acima de 20 °C e abaixo de 15 °C (também separados para dias úteis), máxima acima de 28 °C, temperatura da semana de referência e diferença para ela | a do dia previsto viria da previsão do tempo em produção, e aqui usamos a observada. A da semana de referência já aconteceu |
| Perfil EPE | participação residencial, industrial e comercial no PR e crescimento do consumo em 12 meses, com 2 meses de atraso | sim, o atraso respeita a publicação da EPE |

In [ ]:
def criar_variaveis(d, h=HORIZONTE_DIAS):
    """Monta as variáveis candidatas e a coluna de origem de cada uma."""
    f = pd.DataFrame(index=d.index)
    origem = {}
    carga = d["carga_mwmed"]
    sem = int(np.ceil(h / 7) * 7)          # mesmo dia da semana

    # calendário
    for k in range(1, 7):
        f[f"dow_{k}"] = (d["dia_semana"] == k).astype(int); origem[f"dow_{k}"] = "dia_semana"
    doy = d.index.dayofyear
    f["mes_sen"], f["mes_cos"] = np.sin(2 * np.pi * d["mes"] / 12), np.cos(2 * np.pi * d["mes"] / 12)
    f["ano_sen"], f["ano_cos"] = np.sin(2 * np.pi * doy / 365.25), np.cos(2 * np.pi * doy / 365.25)
    origem.update({"mes_sen": "mes", "mes_cos": "mes", "ano_sen": "data", "ano_cos": "data"})
    fer = d["feriado"].fillna(0).astype(int)
    f["feriado"] = fer
    f["vespera_feriado"] = fer.shift(-1, fill_value=0)
    f["pos_feriado"] = fer.shift(1, fill_value=0)
    f["fim_de_ano"] = (((d.index.month == 12) & (d.index.day >= 24)) | ((d.index.month == 1) & (d.index.day <= 2))).astype(int)
    origem.update({"feriado": "feriado", "vespera_feriado": "feriado", "pos_feriado": "feriado", "fim_de_ano": "data"})
    for r in (2, 3):
        f[f"regime_{r}"] = (d["regime_metodologico"] == r).astype(int); origem[f"regime_{r}"] = "regime_metodologico"
    f["tendencia_anos"] = (d.index - pd.Timestamp("2015-01-01")).days / 365.25; origem["tendencia_anos"] = "data"

    # histórico da carga até D
    f["nivel_semana"] = carga.shift(h).rolling(7, min_periods=7).mean()
    for k, lag in (("1", sem), ("2", sem + 7), ("3", sem + 14)):
        f[f"carga_sem{k}"] = carga.shift(lag)
    f["carga_ano"] = carga.shift(364)
    for c in ("nivel_semana", "carga_sem1", "carga_sem2", "carga_sem3", "carga_ano"):
        origem[c] = "carga_mwmed"
    # versão relativa
    for c in ("carga_sem1", "carga_sem2", "carga_sem3", "carga_ano"):
        f[f"rel_{c}"] = f[c] / f["nivel_semana"]; origem[f"rel_{c}"] = "carga_mwmed"

    # temperatura
    for c in ("temp_media_sul", "temp_max_sul", "temp_min_sul", "graus_dia_calor", "graus_dia_frio"):
        f[c] = d[c]; origem[c] = c
    t = d["temp_media_sul"]
    util = ((d["dia_semana"] < 5) & (fer == 0)).astype(int)
    f["calor_20"] = (t - 20).clip(lower=0)
    f["frio_15"] = (15 - t).clip(lower=0)
    f["max_acima_28"] = (d["temp_max_sul"] - 28).clip(lower=0)
    f["calor_20_util"] = f["calor_20"] * util
    f["frio_15_util"] = f["frio_15"] * util
    # temperatura da semana de referência
    f["temp_semana_ref"] = t.shift(h).rolling(7, min_periods=7).mean()
    f["dif_temp_ref"] = t - f["temp_semana_ref"]
    f["dif_calor_ref"] = f["calor_20"] - f["calor_20"].shift(h).rolling(7, min_periods=7).mean()
    f["dif_frio_ref"] = f["frio_15"] - f["frio_15"].shift(h).rolling(7, min_periods=7).mean()
    for c in ("calor_20", "frio_15", "calor_20_util", "frio_15_util", "temp_semana_ref", "dif_temp_ref",
              "dif_calor_ref", "dif_frio_ref"):
        origem[c] = "temp_media_sul"
    origem["max_acima_28"] = "temp_max_sul"

    # perfil da EPE, já defasado
    for c in ("part_residencial_pr_lag2m", "part_industrial_pr_lag2m", "part_comercial_pr_lag2m"):
        f[c] = d[c]; origem[c] = c
    f["cresc_consumo_12m_lag2m"] = d["consumo_pr_total_mwh_lag2m"] / d["consumo_pr_total_mwh_lag2m"].shift(365) - 1
    origem["cresc_consumo_12m_lag2m"] = "consumo_pr_total_mwh_lag2m"
    return f, origem

X_todas, ORIGEM = criar_variaveis(prep)

CALENDARIO = [f"dow_{k}" for k in range(1, 7)] + ["mes_sen", "mes_cos", "ano_sen", "ano_cos", "feriado",
              "vespera_feriado", "pos_feriado", "fim_de_ano"]
GRUPOS = {
    "calendario": CALENDARIO + ["regime_2", "regime_3", "tendencia_anos"],
    "calendario_relativo": CALENDARIO,
    "regime": ["regime_2", "regime_3"],
    "historico": ["nivel_semana", "carga_sem1", "carga_sem2", "carga_sem3", "carga_ano"],
    "historico_relativo": ["rel_carga_sem1", "rel_carga_sem2", "rel_carga_sem3", "rel_carga_ano"],
    "temperatura": ["temp_media_sul", "temp_max_sul", "temp_min_sul", "graus_dia_calor", "graus_dia_frio",
                    "calor_20", "frio_15", "max_acima_28", "calor_20_util", "frio_15_util",
                    "temp_semana_ref", "dif_temp_ref", "dif_calor_ref", "dif_frio_ref"],
    "epe": ["part_residencial_pr_lag2m", "part_industrial_pr_lag2m", "part_comercial_pr_lag2m", "cresc_consumo_12m_lag2m"],
}
FONTE_DO_GRUPO = {"temperatura": "Open-Meteo", "epe": "EPE"}
validar_variaveis([ORIGEM[c] for c in X_todas.columns] + ["feriado", "nome_feriado"])

# feriados para o Prophet
_fer = prep.loc[prep["feriado"] == 1, ["nome_feriado"]].reset_index()
FERIADOS_PROPHET = pd.DataFrame({"holiday": _fer["nome_feriado"].fillna("feriado").to_numpy(), "ds": _fer["data"].to_numpy(),
                                 "lower_window": -1, "upper_window": 1})   # véspera e dia seguinte
print(f"{X_todas.shape[1]} variáveis candidatas, todas aprovadas pela barreira de vazamento")
cobertura = X_todas.loc[prep["janela"] != "fora"].notna().mean().mul(100).round(1)
print("Variáveis com lacuna nas janelas de modelagem:", cobertura[cobertura < 100].to_dict() or "nenhuma")

## 6. Métricas e justificativa pelo negócio

| Métrica | O que mede | Por que importa para a Copel |
|---|---|---|
| MAE (MWmed) | erro médio absoluto na unidade do planejamento | é o número que o gestor entende, quanto erramos em média por dia. É a métrica principal de escolha |
| RMSE (MWmed) | erro que pesa mais os erros grandes | um erro grande isolado é o que causa sobrecarga ou compra de energia errada |
| MAPE (%) | erro médio em percentual da carga | compara períodos com níveis de carga diferentes, já que a carga cresce ano a ano |
| Viés (MWmed) | média de previsto menos real | negativo quer dizer que o modelo subestima, o que é mais grave (risco de sobrecarga e de piorar DEC/FEC) do que superestimar (custo) |
| MAE nos picos | MAE só nos 10% de dias de maior carga | a capacidade é dimensionada pelos picos |
| Ganho sobre a referência | quanto o MAE cai em relação à regra simples | o modelo só se justifica se for melhor que repetir a semana passada |

In [ ]:
def metricas(real, prev, prefixo):
    real, prev = pd.Series(real), pd.Series(prev, index=real.index)
    erro = prev - real
    pico = real >= real.quantile(PERCENTIL_PICO)
    return {
        f"MAE_{prefixo}": erro.abs().mean(),
        f"RMSE_{prefixo}": np.sqrt((erro ** 2).mean()),
        f"MAPE_{prefixo}": (erro.abs() / real).mean() * 100,
        f"vies_{prefixo}": erro.mean(),
        f"MAE_pico_{prefixo}": erro[pico].abs().mean(),
        f"erro_max_{prefixo}": erro.abs().max(),
    }

## 7. Treinamento, avaliação e registro de cada run

A função `executar_run` é o pipeline inteiro para uma configuração. Ela escolhe as variáveis, passa pela barreira de vazamento, separa as janelas, treina, avalia e grava uma linha no registro de experimentos. Assim todas as runs usam os mesmos dados, as mesmas janelas e as mesmas métricas.

O hash do código do pipeline (texto das funções acima) também é gravado. Se alguém mudar a forma de criar variáveis, por exemplo, as runs novas ficam com outra versão de código e não se confundem com as antigas.

In [ ]:
class ReferenciaSazonal:
    """Repete a carga do mesmo dia da semana anterior."""
    def __init__(self, coluna="carga_sem1"):
        self.coluna = coluna
    def fit(self, X, y):
        return self
    def predict(self, X):
        return X[self.coluna].to_numpy()
    def get_params(self, deep=False):
        return {"coluna": self.coluna}

class ProphetRegressor:
    """Prophet com a interface do scikit-learn."""
    def __init__(self, sazonalidade="multiplicative", changepoint_prior_scale=0.05):
        self.sazonalidade, self.changepoint_prior_scale = sazonalidade, changepoint_prior_scale
    def get_params(self, deep=False):
        return {"sazonalidade": self.sazonalidade, "changepoint_prior_scale": self.changepoint_prior_scale,
                "feriados": f"{FERIADOS_PROPHET['holiday'].nunique()} tipos, janela -1 a +1 dia"}
    def _tabela(self, X):
        df = pd.DataFrame({"ds": X.index})
        for c in X.columns:
            df[c] = X[c].to_numpy()
        return df
    def fit(self, X, y):
        self.colunas = list(X.columns)
        self.m = Prophet(holidays=FERIADOS_PROPHET, seasonality_mode=self.sazonalidade,
                         changepoint_prior_scale=self.changepoint_prior_scale,
                         yearly_seasonality=True, weekly_seasonality=True, daily_seasonality=False)
        for c in self.colunas:
            self.m.add_regressor(c)
        self.m.fit(self._tabela(X).assign(y=np.asarray(y)))
        return self
    def componentes(self, X):
        return self.m.predict(self._tabela(X[self.colunas])).set_index("ds")
    def predict(self, X):
        return self.componentes(X)["yhat"].to_numpy()

ALGORITMOS = {
    "Referência sazonal": ReferenciaSazonal,
    "Regressão Linear": LinearRegression,
    "Random Forest": RandomForestRegressor,
    "Gradient Boosting": HistGradientBoostingRegressor,
    "Prophet": ProphetRegressor,
}
COMPLEXIDADE = {"Referência sazonal": 0, "Regressão Linear": 1, "Random Forest": 2, "Gradient Boosting": 2, "Prophet": 2}
# runs de diagnóstico, fora da escolha
DIAGNOSTICO = {"R04": "erro de desenho (indicador de regime no modelo relativo)",
               "R14": "sem retreino, não usa nenhum dado posterior a 2023"}

def _hash_texto(t):
    return hashlib.sha256(t.encode("utf-8")).hexdigest()[:12]

def _fonte(f):
    try:
        return inspect.getsource(f)
    except (OSError, TypeError):          # fora do Jupyter
        return f.__code__.co_code.hex() + repr(f.__code__.co_consts)

VERSAO_PIPELINE = _hash_texto("".join(_fonte(f) for f in
                                      (carregar_dados_brutos, validar_entrada, validar_variaveis, preparar, criar_variaveis, metricas)))

registro, previsoes, modelos_treinados = [], {}, {}

def executar_run(run_id, descricao, algoritmo, parametros, grupos, alvo="nivel", excluir_treino=(), retreino=None,
                 treino=("treino",), avaliar=("validacao", "teste"), salvar_modelo=False):
    inicio = time.time()
    variaveis = [c for g in grupos for c in GRUPOS[g]]
    if algoritmo == "Referência sazonal":
        variaveis = ["carga_sem1"]
    validar_variaveis([ORIGEM[c] for c in variaveis])

    base = X_todas[variaveis].copy()
    if alvo == "razao":
        y = prep["carga_mwmed"] / X_todas["nivel_semana"]
    else:
        y = prep["carga_mwmed"]
    usavel = prep["avaliavel"] & base.notna().all(axis=1) & X_todas["nivel_semana"].notna()

    no_treino = usavel & prep["janela"].isin(treino)
    excluidos, fora_sempre = {}, pd.Series(False, index=prep.index)
    for flag in ("historico_cruza_regime",) + tuple(excluir_treino):
        marca = no_treino & (prep[flag] == 1)
        excluidos[flag] = int(marca.sum())
        no_treino &= ~marca
        fora_sempre |= prep[flag] == 1

    params = dict(parametros)
    if algoritmo in ("Random Forest", "Gradient Boosting"):
        params.setdefault("random_state", SEMENTE)
    modelo = ALGORITMOS[algoritmo](**params)
    modelo.fit(base[no_treino], y[no_treino])
    tempo = time.time() - inicio

    linha = {
        "id_execucao": ID_EXECUCAO, "run": run_id, "descricao": descricao,
        "data_execucao": DATA_EXECUCAO.strftime("%Y-%m-%d %H:%M:%S"),
        "versao_dados": ROTULO_DADOS, "versao_pipeline": VERSAO_PIPELINE,
        "modelo": algoritmo, "parametros": json.dumps(params, ensure_ascii=False),
        "parametros_completos": json.dumps(modelo.get_params(), ensure_ascii=False, default=str),
        "alvo": "carga (MWmed)" if alvo == "nivel" else "razão carga / média da semana conhecida",
        "grupos_variaveis": "+".join(grupos) if algoritmo != "Referência sazonal" else "carga de 7 dias antes",
        "fontes_externas": "+".join(sorted({FONTE_DO_GRUPO[g] for g in grupos if g in FONTE_DO_GRUPO})) or "nenhuma",
        "n_variaveis": len(variaveis),
        "janela_treino": " + ".join(f"{JANELAS[j][0]} a {JANELAS[j][1]}" for j in treino),
        "dias_treino": int(no_treino.sum()),
        "excluidos_do_treino": json.dumps(excluidos),
        "horizonte_dias": HORIZONTE_DIAS, "semente": SEMENTE if "random_state" in params else None,
        "tempo_treino_s": round(tempo, 2),
        "retreino": "nenhum" if retreino is None else retreino,
        "elegivel_escolha": run_id not in DIAGNOSTICO,
        "observacao": DIAGNOSTICO.get(run_id, ""),
    }
    ajustes = 1
    for janela in avaliar:
        m = usavel & (prep["janela"] == janela)
        if retreino == "mensal":
            # retreino mensal com dados até 7 dias antes do mês
            p = pd.Series(index=prep.index[m], dtype=float)
            apto = usavel & ~fora_sempre & (prep.index >= JANELAS[treino[0]][0])
            for mes, dias in pd.Series(p.index, index=p.index).groupby(p.index.to_period("M")).groups.items():
                origem = mes.start_time - pd.Timedelta(days=HORIZONTE_DIAS)
                m_tr = apto & (prep.index <= origem)
                mod_mes = ALGORITMOS[algoritmo](**params).fit(base[m_tr], y[m_tr])
                p[dias] = mod_mes.predict(base.loc[dias])
                ajustes += 1
            p = p.to_numpy()
        else:
            p = modelo.predict(base[m])
        if alvo == "razao":
            p = p * X_todas.loc[m, "nivel_semana"].to_numpy()
        real = prep.loc[m, "carga_mwmed"]
        linha.update(metricas(real, p, janela))
        linha[f"dias_{janela}"] = int(m.sum())
        previsoes[(run_id, janela)] = pd.Series(p, index=real.index)
    if retreino:
        linha["tempo_treino_s"] = round(time.time() - inicio, 2)
        linha["n_ajustes"] = ajustes
    ref = [r for r in registro if r["run"] == "R01"]
    for janela in avaliar:
        if ref and f"MAE_{janela}" in ref[0]:
            linha[f"ganho_vs_ref_{janela}_pct"] = (1 - linha[f"MAE_{janela}"] / ref[0][f"MAE_{janela}"]) * 100
    if salvar_modelo:
        pacote = {"modelo": modelo, "variaveis": variaveis, "alvo": alvo, "horizonte": HORIZONTE_DIAS,
                  "retreino": retreino, "versao_dados": ROTULO_DADOS, "versao_pipeline": VERSAO_PIPELINE}
        arq = MODELOS / f"{run_id}.joblib"
        try:
            joblib.dump(pacote, arq)
        except Exception:                 # Prophet vai em JSON
            from prophet.serialize import model_to_json
            arq = MODELOS / f"{run_id}_prophet.json"
            arq.write_text(model_to_json(modelo.m))
        linha["arquivo_modelo"] = arq.name
        linha["sha256_modelo"] = sha256(arq)[:12]
    registro.append(linha)
    modelos_treinados[run_id] = (modelo, variaveis, alvo, base[no_treino].mean())
    mostrar = {k: linha[k] for k in linha if k.startswith(("MAE_", "MAPE_", "ganho"))}
    print(f"{run_id} {descricao}: " + ", ".join(f"{k}={v:,.1f}" for k, v in mostrar.items()))
    return linha

### 7.1 Runs planejadas

Cada run muda uma coisa em relação a outra, para que a diferença de erro possa ser atribuída a essa mudança.

| Run | O que testa |
|---|---|
| R01 | referência, que repete a carga do mesmo dia da semana anterior |
| R02 e R03 | Regressão Linear absoluta, sem e com temperatura |
| R04 e R05 | Regressão Linear relativa, com e sem o indicador de regime (erro de desenho detectado pelo registro) |
| R06 e R07 | Random Forest com alvo absoluto e com alvo relativo (problema de extrapolação) |
| R08 | Random Forest relativo sem temperatura |
| R09 | R07 com perfil de consumo da EPE |
| R10 | Gradient Boosting relativo |
| R11 | R05 treinado sem os dias de grande interrupção (demanda reprimida por apagão) |
| R12 | R11 sem também os eventos conhecidos (greve de 2018, pandemia em 2020) |
| R13 | R07 com árvores mais rasas |
| R14 | Prophet treinado uma vez (até 2023), sem retreino |
| R15 | Prophet com retreino mensal (cada mês é previsto por um modelo treinado com dados até 7 dias antes) |
| R16 | R15 com sazonalidade aditiva em vez de multiplicativa |

O Prophet, citado na Etapa 1, modela a série como tendência, sazonalidade semanal, sazonalidade anual, feriados e regressores (aqui a temperatura). Ele não recebe a carga recente como entrada, então a forma de usar os dados mais novos é treinar de novo. A R14 mostra o que acontece sem isso. As runs R04 e R14 ficam no registro só para documentar problemas e não podem ser escolhidas.

Em todas as runs os dias de transição de metodologia do ONS ficam fora do treino, e isso fica na coluna `excluidos_do_treino`.

In [ ]:
RF = {"n_estimators": 300, "min_samples_leaf": 3, "max_features": 0.5, "n_jobs": -1}
GB = {"max_iter": 400, "learning_rate": 0.05, "max_leaf_nodes": 15, "min_samples_leaf": 10}
PR = {"sazonalidade": "multiplicative", "changepoint_prior_scale": 0.05}
REL = ["calendario_relativo", "historico_relativo"]
ABS = ["calendario", "historico"]

PLANO = [
    ("R01", "Referência sazonal (semana anterior)", "Referência sazonal", {}, [], "nivel", ()),
    ("R02", "Regressão Linear absoluta, sem temperatura", "Regressão Linear", {}, ABS, "nivel", ()),
    ("R03", "Regressão Linear absoluta + temperatura", "Regressão Linear", {}, ABS + ["temperatura"], "nivel", ()),
    ("R04", "Regressão Linear relativa + temperatura, com regime", "Regressão Linear", {},
     REL + ["regime", "temperatura"], "razao", ()),
    ("R05", "Regressão Linear relativa + temperatura", "Regressão Linear", {}, REL + ["temperatura"], "razao", ()),
    ("R06", "Random Forest absoluto + temperatura", "Random Forest", RF, ABS + ["temperatura"], "nivel", ()),
    ("R07", "Random Forest relativo + temperatura", "Random Forest", RF, REL + ["temperatura"], "razao", ()),
    ("R08", "Random Forest relativo, sem temperatura", "Random Forest", RF, REL, "razao", ()),
    ("R09", "Random Forest relativo + temperatura + EPE", "Random Forest", RF, REL + ["temperatura", "epe"], "razao", ()),
    ("R10", "Gradient Boosting relativo + temperatura", "Gradient Boosting", GB, REL + ["temperatura"], "razao", ()),
    ("R11", "R05 sem dias de grande interrupção no treino", "Regressão Linear", {}, REL + ["temperatura"], "razao",
     ("flag_grande_interrupcao",)),
    ("R12", "R05 sem grandes interrupções e eventos conhecidos", "Regressão Linear", {}, REL + ["temperatura"], "razao",
     ("flag_grande_interrupcao", "flag_evento_conhecido")),
    ("R13", "R07 com árvores mais rasas", "Random Forest", {**RF, "max_depth": 8, "min_samples_leaf": 10},
     REL + ["temperatura"], "razao", ()),
    ("R14", "Prophet sem retreino", "Prophet", PR, ["temperatura"], "nivel", ()),
    ("R15", "Prophet com retreino mensal", "Prophet", PR, ["temperatura"], "nivel", (), "mensal"),
    ("R16", "Prophet com retreino mensal, sazonalidade aditiva", "Prophet", {**PR, "sazonalidade": "additive"},
     ["temperatura"], "nivel", (), "mensal"),
]
for plano in PLANO:
    executar_run(*plano)

### 7.2 Escolha do modelo (só com a validação de 2024)

A regra foi definida antes de olhar o teste. Vence o menor MAE na validação. Se outras runs ficarem a até 1% desse MAE, consideramos empate técnico e preferimos a que depende de menos fontes externas e, depois, o modelo mais simples (referência, depois Regressão Linear, depois árvores e Prophet). Cada fonte e cada camada de complexidade a mais traz risco de falha, custo de manutenção e dificuldade para explicar a previsão. Persistindo o empate, vale o menor MAE. Runs de diagnóstico não entram.

In [ ]:
reg = pd.DataFrame(registro)
candidatos = reg[(reg["run"] != "R01") & reg["elegivel_escolha"]].copy()
melhor_mae = candidatos["MAE_validacao"].min()
empatados = candidatos[candidatos["MAE_validacao"] <= melhor_mae * (1 + TOLERANCIA_EMPATE)].copy()
empatados["n_fontes"] = empatados["fontes_externas"].map(lambda s: 0 if s == "nenhuma" else len(s.split("+")))
empatados["complexidade"] = empatados["modelo"].map(COMPLEXIDADE)
escolhida = empatados.sort_values(["n_fontes", "complexidade", "MAE_validacao"]).iloc[0]
RUN_ESCOLHIDA = escolhida["run"]
print(f"Menor MAE de validação: {melhor_mae:,.1f} MWmed")
print("Empate técnico (até 1%) entre:", ", ".join(f"{r} ({v:,.0f})" for r, v in
      empatados.sort_values("MAE_validacao")[["run", "MAE_validacao"]].itertuples(index=False)))
print(f"Escolhida: {RUN_ESCOLHIDA} - {escolhida['descricao']}")

colunas_vis = ["run", "descricao", "fontes_externas", "MAE_validacao", "MAPE_validacao", "MAE_teste", "MAPE_teste",
               "RMSE_teste", "vies_teste", "MAE_pico_teste", "ganho_vs_ref_teste_pct"]
reg[colunas_vis].round(1)

### 7.3 Modelo final

A configuração escolhida é treinada de novo com treino e validação (2017 a 2024), porque os dados mais recentes são os mais parecidos com o futuro. O teste de 2025 continua intocado e serve de confirmação. O modelo final é salvo em arquivo com o hash registrado.

In [ ]:
cfg = next(p for p in PLANO if p[0] == RUN_ESCOLHIDA)
final = executar_run("FINAL", f"{cfg[1]} (treino 2017-2024)", *cfg[2:],
                     treino=("treino", "validacao"), avaliar=("teste",), salvar_modelo=True)
reg = pd.DataFrame(registro)
ref = reg.loc[reg["run"] == "R01"].iloc[0]
print(f"\nTeste 2025 - modelo final: MAE {final['MAE_teste']:,.0f} MWmed ({final['MAPE_teste']:.2f}%), "
      f"referência: MAE {ref['MAE_teste']:,.0f} MWmed ({ref['MAPE_teste']:.2f}%)")

## 8. Avaliação detalhada do modelo final

In [ ]:
real_teste = prep.loc[previsoes[("FINAL", "teste")].index, "carga_mwmed"]
prev_final = previsoes[("FINAL", "teste")]
prev_ref = previsoes[("R01", "teste")].reindex(real_teste.index)
aval = pd.DataFrame({"real": real_teste, "previsto": prev_final, "referencia": prev_ref})
aval["erro"] = aval["previsto"] - aval["real"]
aval["erro_pct"] = aval["erro"] / aval["real"] * 100
aval["dia_semana"] = prep.loc[aval.index, "dia_semana"]
aval["feriado"] = prep.loc[aval.index, "feriado"]
aval["grande_interrupcao"] = prep.loc[aval.index, "flag_grande_interrupcao"]

por_mes = aval.groupby(aval.index.month).agg(MAE=("erro", lambda e: e.abs().mean()),
                                             MAPE=("erro_pct", lambda e: e.abs().mean()), vies=("erro", "mean"))
por_mes.index.name = "mes"
tipos = {"dias úteis comuns": (aval["feriado"] == 0) & (aval["dia_semana"] < 5),
         "fins de semana": (aval["feriado"] == 0) & (aval["dia_semana"] >= 5),
         "feriados": aval["feriado"] == 1,
         "dias de grande interrupção": aval["grande_interrupcao"] == 1}
por_tipo = pd.DataFrame({n: {"dias": int(m.sum()), "MAE": aval.loc[m, "erro"].abs().mean(),
                             "MAPE": aval.loc[m, "erro_pct"].abs().mean(),
                             "MAE_referencia": (aval.loc[m, "referencia"] - aval.loc[m, "real"]).abs().mean()}
                         for n, m in tipos.items()}).T
subestimou_5 = int((aval["erro_pct"] < -5).sum())
piores = aval.reindex(aval["erro"].abs().sort_values(ascending=False).index).head(8)
piores = piores.assign(feriado_nome=prep.loc[piores.index, "nome_feriado"].fillna(""))
print(f"Dias em que o modelo subestimou a carga em mais de 5%: {subestimou_5} de {len(aval)}")
display(por_tipo.round(1))
display(por_mes.round(1).T)
piores[["real", "previsto", "erro", "erro_pct", "feriado_nome", "grande_interrupcao"]].round(1)

### 8.1 Quais variáveis o modelo mais usa

Importância por permutação na validação. Embaralhamos uma variável de cada vez e medimos quanto o erro piora. Quanto maior a piora, mais o modelo depende daquela informação. Ajuda a explicar o modelo para quem não é técnico, como pede a transparência da Etapa 1.

In [ ]:
modelo_e, vars_e, alvo_e, _ = modelos_treinados[RUN_ESCOLHIDA]
m_val = prep["avaliavel"] & (prep["janela"] == "validacao") & X_todas[vars_e].notna().all(axis=1)
y_val = prep.loc[m_val, "carga_mwmed"] / (X_todas.loc[m_val, "nivel_semana"] if alvo_e == "razao" else 1)
imp = permutation_importance(modelo_e, X_todas.loc[m_val, vars_e], y_val, n_repeats=10, random_state=SEMENTE,
                             scoring="neg_mean_absolute_error")
escala = X_todas.loc[m_val, "nivel_semana"].mean() if alvo_e == "razao" else 1
importancia = (pd.DataFrame({"variavel": vars_e, "aumento_MAE_mwmed": imp.importances_mean * escala})
               .sort_values("aumento_MAE_mwmed", ascending=False).reset_index(drop=True))
importancia.head(12).round(1)

### 8.2 Explicabilidade (XAI)

A Etapa 1 prometeu previsões compreensíveis para gestores não técnicos. A importância por permutação mostra o que o modelo usa em geral. Aqui explicamos cada previsão e a sensibilidade à temperatura com três técnicas.

1. Explicação de dias específicos. Partimos de um dia típico (todas as variáveis na média do treino, com o nível de carga da semana de referência) e medimos quanto cada grupo de informação (calendário, histórico, temperatura) move a previsão em MWmed. Na Regressão Linear a conta é exata e as parcelas somam a previsão. Em modelos não lineares sobra uma parcela de interação, mostrada à parte.
2. Sensibilidade. Aumentamos em 1 °C a temperatura prevista de todos os dias de 2025 e medimos quanto a previsão muda, por faixa de temperatura.
3. Componentes do Prophet. O próprio Prophet separa a previsão em tendência, dia da semana, época do ano, feriados e temperatura.

In [ ]:
NOME_GRUPO = {"calendario": "calendário", "calendario_relativo": "calendário", "regime": "regime do ONS",
              "historico": "histórico recente da carga", "historico_relativo": "histórico recente da carga",
              "temperatura": "temperatura", "epe": "perfil de consumo EPE"}
modelo_f, vars_f, alvo_f, medias_f = modelos_treinados["FINAL"]
grupos_f = cfg[4]
grupo_da_var = {v: NOME_GRUPO[g] for g in grupos_f for v in GRUPOS[g]}

def prever_final(Xp):
    p = modelo_f.predict(Xp[vars_f])
    return p * X_todas.loc[Xp.index, "nivel_semana"].to_numpy() if alvo_f == "razao" else p

def explicar_dias(datas):
    Xd = X_todas.loc[datas, vars_f]
    previsto = prever_final(Xd)
    tipico_X = pd.DataFrame([medias_f.values] * len(Xd), index=Xd.index, columns=vars_f)
    tipico = prever_final(tipico_X)
    linhas = {"dia típico": tipico}
    for nome in dict.fromkeys(grupo_da_var.values()):
        cols = [v for v in vars_f if grupo_da_var[v] == nome]
        sem_grupo = Xd.copy(); sem_grupo[cols] = medias_f[cols].values
        linhas[nome] = previsto - prever_final(sem_grupo)
    tab = pd.DataFrame(linhas, index=Xd.index)
    tab["interação"] = previsto - tab.sum(axis=1)
    tab["previsto"] = previsto
    tab["real"] = prep.loc[datas, "carga_mwmed"].to_numpy()
    return tab

ano_teste = aval.index
domingos = aval[(aval["dia_semana"] == 6) & (aval["feriado"] == 0)]
fer_util = aval[(aval["feriado"] == 1) & (aval["dia_semana"] < 5)]
EXEMPLOS = {
    "maior carga do ano": aval["real"].idxmax(),
    "dia mais frio": prep.loc[ano_teste, "temp_media_sul"].idxmin(),
    "domingo comum": (domingos["real"] - domingos["real"].median()).abs().idxmin(),
    "feriado em dia útil": fer_util["real"].idxmin() if len(fer_util) else aval.index[0],
}
explicacao = explicar_dias(list(EXEMPLOS.values()))
explicacao.insert(0, "exemplo", list(EXEMPLOS.keys()))
explicacao.insert(1, "temperatura_c", prep.loc[explicacao.index, "temp_media_sul"].round(1).to_numpy())
nome_fer = prep.loc[explicacao.index, "nome_feriado"].fillna("")
explicacao.insert(2, "feriado", nome_fer.to_numpy())
print("Parcelas em MWmed (dia típico + parcelas = previsto)")
explicacao.round(0)

In [ ]:
# sensibilidade a +1 °C no dia previsto
def mais_temperatura(Xb, delta):
    d = prep.loc[Xb.index]
    t, tmax, tmin = d["temp_media_sul"] + delta, d["temp_max_sul"] + delta, d["temp_min_sul"] + delta
    util = ((d["dia_semana"] < 5) & (d["feriado"].fillna(0) == 0)).astype(int)
    calor, frio = (t - 20).clip(lower=0), (15 - t).clip(lower=0)
    ref = X_todas.loc[Xb.index]
    novos = {"temp_media_sul": t, "temp_max_sul": tmax, "temp_min_sul": tmin,
             "graus_dia_calor": (t - 24).clip(lower=0), "graus_dia_frio": (18 - t).clip(lower=0),
             "calor_20": calor, "frio_15": frio, "max_acima_28": (tmax - 28).clip(lower=0),
             "calor_20_util": calor * util, "frio_15_util": frio * util,
             "dif_temp_ref": t - ref["temp_semana_ref"],
             "dif_calor_ref": calor - (ref["calor_20"] - ref["dif_calor_ref"]),
             "dif_frio_ref": frio - (ref["frio_15"] - ref["dif_frio_ref"])}
    Xn = Xb.copy()
    for c, v in novos.items():
        if c in Xn:
            Xn[c] = v.to_numpy()
    return Xn

if "temperatura" in grupos_f:
    X_teste = X_todas.loc[aval.index, vars_f]
    efeito = pd.DataFrame({"delta_mwmed": prever_final(mais_temperatura(X_teste, 1.0)) - prever_final(X_teste),
                           "temp": prep.loc[aval.index, "temp_media_sul"],
                           "tipo": np.where((aval["dia_semana"] < 5) & (aval["feriado"] == 0), "dia útil", "fim de semana/feriado")},
                          index=aval.index)
    efeito["delta_pct"] = efeito["delta_mwmed"] / aval["previsto"] * 100
    efeito["faixa"] = pd.cut(efeito["temp"], [-np.inf, 15, 20, 24, np.inf],
                             labels=["abaixo de 15 °C", "15 a 20 °C", "20 a 24 °C", "acima de 24 °C"])
    sensibilidade = (efeito.groupby(["faixa", "tipo"], observed=True)
                     .agg(dias=("delta_mwmed", "size"), mwmed_por_grau=("delta_mwmed", "mean"), pct_por_grau=("delta_pct", "mean")))
    display(sensibilidade.round(2))
else:
    sensibilidade = pd.DataFrame()
    print("O modelo escolhido não usa temperatura.")

In [ ]:
# componentes do Prophet (R15)
cfg_p = next(p for p in PLANO if p[0] == "R15")
vars_p = [c for g in cfg_p[4] for c in GRUPOS[g]]
m_tr = (prep["avaliavel"] & prep["janela"].isin(["treino", "validacao"]) & (prep["historico_cruza_regime"] == 0)
        & X_todas[vars_p].notna().all(axis=1))
prophet_xai = ProphetRegressor(**cfg_p[3]).fit(X_todas.loc[m_tr, vars_p], prep.loc[m_tr, "carga_mwmed"])
comp_hist = prophet_xai.componentes(X_todas.loc[m_tr | (prep["janela"] == "teste"), vars_p])
comp = comp_hist.loc[aval.index.min():aval.index.max()]
multiplicativo = cfg_p[3].get("sazonalidade") == "multiplicative"
fator = 100 if multiplicativo else 1
unidade = "% da tendência" if multiplicativo else "MWmed"
efeito_semana = (comp["weekly"].groupby(comp.index.dayofweek).mean() * fator)
efeito_semana.index = ["seg", "ter", "qua", "qui", "sex", "sáb", "dom"]
efeito_fer = (comp.loc[comp["holidays"] != 0, "holidays"] * fator).rename("efeito")
efeito_fer = efeito_fer.to_frame().assign(feriado=prep.loc[efeito_fer.index, "nome_feriado"].fillna("(véspera ou dia seguinte)"))
print(f"Prophet - efeitos em {unidade}")
print("Dia da semana:", efeito_semana.round(1).to_dict())
print(f"Época do ano: de {comp['yearly'].min() * fator:.1f} a {comp['yearly'].max() * fator:.1f}")
print(f"Temperatura: de {comp['extra_regressors_' + ('multiplicative' if multiplicativo else 'additive')].min() * fator:.1f} "
      f"a {comp['extra_regressors_' + ('multiplicative' if multiplicativo else 'additive')].max() * fator:.1f}")
print(f"Tendência: {comp_hist['trend'].iloc[0]:,.0f} MWmed em {comp_hist.index[0]:%m/%Y} e {comp['trend'].iloc[-1]:,.0f} MWmed em {comp.index[-1]:%m/%Y}")
efeito_fer.sort_values("efeito").head(8).round(1)

## 9. Figuras

In [ ]:
# figura 1, erro por run
reg_plot = reg[reg["run"] != "FINAL"].sort_values("MAE_validacao", ascending=True)
fig, ax = plt.subplots(figsize=(11, 6.2))
y = np.arange(len(reg_plot))
ax.barh(y - 0.2, reg_plot["MAE_validacao"], 0.4, color=COR_SERIE, label="validação 2024 (escolha)")
ax.barh(y + 0.2, reg_plot["MAE_teste"], 0.4, color=COR_2, label="teste 2025 (confirmação)")
ax.set_yticks(y, [f"{r}  {d}" + (" (diagnóstico)" if r in DIAGNOSTICO else "")
                  for r, d in zip(reg_plot["run"], reg_plot["descricao"])], fontsize=8.5)
ax.invert_yaxis(); ax.set_xlabel("MAE (MWmed) - menor é melhor"); ax.grid(axis="y", visible=False)
i_esc = list(reg_plot["run"]).index(RUN_ESCOLHIDA)
ax.get_yticklabels()[i_esc].set_fontweight("bold")
ax.legend(loc="upper left", bbox_to_anchor=(0, 1.08), ncol=2, frameon=False)
ax.set_title("Erro médio por run (horizonte de 7 dias)", loc="left", pad=26)
fig.savefig(FIG / "fig1_mae_por_run.png", bbox_inches="tight"); plt.show()

# figura 2, real x previsto
fig, ax = plt.subplots(figsize=(12, 4.4))
ax.plot(aval.index, aval["real"], color="#1A1A1A", lw=1.1, label="carga real")
ax.plot(aval.index, aval["referencia"], color=COR_REF, lw=0.9, alpha=0.8, label=f"referência (MAE {ref['MAE_teste']:,.0f})")
ax.plot(aval.index, aval["previsto"], color=COR_2, lw=1.1, label=f"modelo final (MAE {final['MAE_teste']:,.0f})")
ax.set_ylabel("MWmed"); ax.legend(loc="upper left", bbox_to_anchor=(0, 1.13), ncol=3, frameon=False)
ax.set_title("Teste 2025: carga real e prevista com 7 dias de antecedência", loc="left", pad=26)
fig.savefig(FIG / "fig2_real_x_previsto_2025.png", bbox_inches="tight"); plt.show()

# figura 3, erro por mês e tipo de dia
fig, axs = plt.subplots(1, 2, figsize=(12, 4), gridspec_kw={"width_ratios": [1.6, 1]})
nomes_mes = ["jan", "fev", "mar", "abr", "mai", "jun", "jul", "ago", "set", "out", "nov", "dez"]
axs[0].bar([nomes_mes[m - 1] for m in por_mes.index], por_mes["MAPE"], color=COR_SERIE)
axs[0].set_ylabel("MAPE (%)"); axs[0].set_title("Erro por mês (teste 2025)", loc="left"); axs[0].grid(axis="x", visible=False)
pt = por_tipo.dropna(subset=["MAE"])
yy = np.arange(len(pt))
axs[1].barh(yy - 0.2, pt["MAE_referencia"], 0.4, color=COR_REF, label="referência")
axs[1].barh(yy + 0.2, pt["MAE"], 0.4, color=COR_2, label="modelo final")
axs[1].set_yticks(yy, [f"{n} ({int(d)})" for n, d in zip(pt.index, pt["dias"])], fontsize=8.5); axs[1].invert_yaxis()
axs[1].set_xlabel("MAE (MWmed)"); axs[1].grid(axis="y", visible=False)
axs[1].legend(loc="lower right", frameon=True, framealpha=0.9, edgecolor="none")
axs[1].set_title("Erro por tipo de dia (teste 2025)", loc="left")
fig.tight_layout(); fig.savefig(FIG / "fig3_erro_por_mes_e_tipo.png", bbox_inches="tight"); plt.show()

# figura 4, importância
NOMES_LEGIVEIS = {
    "dow_1": "terça-feira", "dow_2": "quarta-feira", "dow_3": "quinta-feira", "dow_4": "sexta-feira",
    "dow_5": "sábado", "dow_6": "domingo", "mes_sen": "mês (seno)", "mes_cos": "mês (cosseno)",
    "ano_sen": "época do ano (seno)", "ano_cos": "época do ano (cosseno)", "feriado": "feriado",
    "vespera_feriado": "véspera de feriado", "pos_feriado": "dia após feriado", "fim_de_ano": "fim de ano",
    "regime_2": "regime ONS 2", "regime_3": "regime ONS 3", "tendencia_anos": "tendência",
    "nivel_semana": "carga média da semana de referência", "carga_sem1": "carga 7 dias antes",
    "carga_sem2": "carga 14 dias antes", "carga_sem3": "carga 21 dias antes", "carga_ano": "carga 1 ano antes",
    "rel_carga_sem1": "carga 7 dias antes (relativa)", "rel_carga_sem2": "carga 14 dias antes (relativa)",
    "rel_carga_sem3": "carga 21 dias antes (relativa)", "rel_carga_ano": "carga 1 ano antes (relativa)",
    "temp_media_sul": "temperatura média", "temp_max_sul": "temperatura máxima", "temp_min_sul": "temperatura mínima",
    "graus_dia_calor": "graus acima de 24 °C", "graus_dia_frio": "graus abaixo de 18 °C",
    "calor_20": "graus acima de 20 °C", "frio_15": "graus abaixo de 15 °C", "max_acima_28": "máxima acima de 28 °C",
    "calor_20_util": "calor em dia útil", "frio_15_util": "frio em dia útil",
    "temp_semana_ref": "temperatura da semana de referência", "dif_temp_ref": "temperatura x semana de referência",
    "dif_calor_ref": "calor x semana de referência", "dif_frio_ref": "frio x semana de referência",
}
importancia["descricao"] = importancia["variavel"].map(NOMES_LEGIVEIS).fillna(importancia["variavel"])
top = importancia.head(10).iloc[::-1]
fig, ax = plt.subplots(figsize=(10, 4.2))
ax.barh(top["descricao"], top["aumento_MAE_mwmed"], color=COR_SERIE)
ax.set_xlabel("quanto o MAE piora ao embaralhar a variável (MWmed)"); ax.grid(axis="y", visible=False)
ax.set_title(f"Variáveis mais usadas pelo modelo escolhido ({RUN_ESCOLHIDA})", loc="left")
fig.savefig(FIG / "fig4_importancia_variaveis.png", bbox_inches="tight"); plt.show()

# figura 5, explicação de dias
partes = [c for c in explicacao.columns if c not in ("exemplo", "temperatura_c", "feriado", "dia típico", "previsto", "real")]
partes = [c for c in partes if c != "interação" or explicacao["interação"].abs().max() > 1]
fig, axs = plt.subplots(1, len(explicacao), figsize=(13, 3.8), sharey=True)
for ax, (data, lin) in zip(axs, explicacao.iterrows()):
    vals = [lin[c] for c in partes]
    ax.barh(partes, vals, color=[COR_2 if v >= 0 else COR_SERIE for v in vals])
    ax.axvline(0, color=COR_REF, lw=0.8); ax.grid(axis="y", visible=False)
    for i, v in enumerate(vals):
        ax.text(v, i, f" {v:+,.0f} ".replace(",", "."), va="center", ha="left" if v >= 0 else "right", fontsize=8)
    lim = max(abs(np.array(vals)).max() * 1.6, 300)
    ax.set_xlim(-lim, lim)
    extra = f" | {lin['feriado']}" if lin["feriado"] else ""
    temp_txt = f"{lin['temperatura_c']:.1f}".replace(".", ",")
    ax.set_title(f"{lin['exemplo']}\n{data:%d/%m/%Y} | {temp_txt} °C{extra}", fontsize=9.5, loc="left")
    ax.set_xlabel(f"típico {lin['dia típico']:,.0f} → previsto {lin['previsto']:,.0f}\nreal {lin['real']:,.0f} MWmed".replace(",", "."), fontsize=8.5)
axs[0].invert_yaxis()
fig.suptitle("Quanto cada informação moveu a previsão (MWmed), modelo final", x=0.01, ha="left", fontweight="bold")
fig.tight_layout(); fig.savefig(FIG / "fig5_explicacao_dias.png", bbox_inches="tight"); plt.show()

# figura 6, sensibilidade
if len(sensibilidade):
    sens = sensibilidade.reset_index()
    fig, ax = plt.subplots(figsize=(9, 3.8))
    faixas = list(sens["faixa"].cat.categories) if hasattr(sens["faixa"], "cat") else sorted(sens["faixa"].unique())
    xx = np.arange(len(faixas))
    for k, (tipo, cor) in enumerate((("dia útil", COR_2), ("fim de semana/feriado", COR_SERIE))):
        sub = sens[sens["tipo"] == tipo].set_index("faixa").reindex(faixas)
        ax.bar(xx + (k - 0.5) * 0.38, sub["mwmed_por_grau"], 0.38, color=cor, label=tipo)
    ax.axhline(0, color=COR_REF, lw=0.8); ax.set_xticks(xx, faixas); ax.grid(axis="x", visible=False)
    ax.set_ylabel("MWmed a mais por +1 °C")
    ax.legend(loc="upper left", frameon=False)
    ax.set_title("Sensibilidade da previsão à temperatura (2025)", loc="left")
    fig.savefig(FIG / "fig6_sensibilidade_temperatura.png", bbox_inches="tight"); plt.show()

# figura 7, componentes do Prophet
fig, axs = plt.subplots(2, 2, figsize=(12, 6.4))
axs[0, 0].plot(comp_hist.index, comp_hist["trend"], color=COR_SERIE)
axs[0, 0].set_title("Tendência (MWmed)", loc="left"); axs[0, 0].axvline(comp.index.min(), color=COR_REF, ls="--", lw=0.8)
axs[0, 1].bar(efeito_semana.index, efeito_semana.values, color=[COR_SERIE] * 5 + [COR_2] * 2)
axs[0, 1].axhline(0, color=COR_REF, lw=0.8); axs[0, 1].set_title(f"Dia da semana ({unidade})", loc="left"); axs[0, 1].grid(axis="x", visible=False)
axs[1, 0].plot(comp.index, comp["yearly"] * fator, color=COR_SERIE)
axs[1, 0].axhline(0, color=COR_REF, lw=0.8); axs[1, 0].set_title(f"Época do ano em 2025 ({unidade})", loc="left")
col_reg = "extra_regressors_" + ("multiplicative" if multiplicativo else "additive")
axs[1, 1].scatter(prep.loc[comp.index, "temp_media_sul"], comp[col_reg] * fator, s=8, color=COR_2, alpha=0.7)
axs[1, 1].axhline(0, color=COR_REF, lw=0.8); axs[1, 1].set_xlabel("temperatura média do dia (°C)")
axs[1, 1].set_title(f"Efeito da temperatura em 2025 ({unidade})", loc="left")
fig.suptitle("Como o Prophet decompõe a previsão", x=0.01, ha="left", fontweight="bold")
fig.tight_layout(); fig.savefig(FIG / "fig7_componentes_prophet.png", bbox_inches="tight"); plt.show()

## 10. Registro de experimentos e saídas

* `registro_experimentos.csv` tem uma linha por run com todos os campos de rastreabilidade. Se o notebook rodar de novo no mesmo ambiente, as novas runs são acrescentadas com outro `id_execucao` e formam o histórico.
* `tabela_runs.csv` é a versão resumida no formato do enunciado (Run, Dados, Modelo, Parâmetros, Métrica).
* `previsoes_teste.csv` tem a previsão de cada run para cada dia de 2025, para auditoria.
* `modelos/FINAL.joblib` é o modelo final, com as variáveis e a versão dos dados.
* `xai_*.csv` têm a explicação de dias específicos, a sensibilidade à temperatura e os componentes do Prophet.

In [ ]:
reg = pd.DataFrame(registro)
if ARQ_REGISTRO.exists():
    anterior = pd.read_csv(ARQ_REGISTRO)
    anterior = anterior[anterior["id_execucao"].astype(str) != ID_EXECUCAO]
    historico = pd.concat([anterior, reg], ignore_index=True)
else:
    historico = reg
historico.round(4).to_csv(ARQ_REGISTRO, index=False)

def resumo_params(r):
    p = json.loads(r["parametros"])
    p = {k: v for k, v in p.items() if k not in ("n_jobs", "random_state")}
    if r["modelo"] == "Referência sazonal":
        return "carga do mesmo dia da semana, 7 dias antes"
    return ", ".join(f"{k}={v}" for k, v in p.items()) or "padrão do scikit-learn"

NOMES_EXCLUSAO = {"historico_cruza_regime": "transição de metodologia", "flag_grande_interrupcao": "grande interrupção",
                  "flag_evento_conhecido": "evento conhecido"}
def resumo_exclusoes(txt):
    return " e ".join(f"{v} dias por {NOMES_EXCLUSAO.get(k, k)}" for k, v in json.loads(txt).items())

tabela_runs = pd.DataFrame({
    "Run": reg["run"],
    "Dados": reg["versao_dados"] + ", " + reg["grupos_variaveis"] + ", treino " + reg["janela_treino"]
             + " (fora do treino " + reg["excluidos_do_treino"].map(resumo_exclusoes) + ")",
    "Modelo": reg["modelo"] + np.where(reg["alvo"].str.startswith("razão"), " (alvo relativo)", ""),
    "Parâmetros": reg.apply(resumo_params, axis=1),
    "MAE validação (MWmed)": reg["MAE_validacao"].round(0),
    "MAE teste (MWmed)": reg["MAE_teste"].round(0),
    "MAPE teste (%)": reg["MAPE_teste"].round(2),
    "Data": reg["data_execucao"],
})
tabela_runs.to_csv(OUT / "tabela_runs.csv", index=False)

prev_tab = pd.DataFrame({"real": real_teste})
for (run_id, janela), s in previsoes.items():
    if janela == "teste":
        prev_tab[run_id] = s
prev_tab.round(1).to_csv(OUT / "previsoes_teste.csv", index_label="data")
tab_validacao.to_csv(OUT / "validacao_entrada.csv", index=False)
importancia.round(2).to_csv(OUT / "importancia_variaveis.csv", index=False)
por_tipo.round(2).to_csv(OUT / "erro_por_tipo_de_dia.csv")
por_mes.round(2).to_csv(OUT / "erro_por_mes.csv")
explicacao.round(1).to_csv(OUT / "xai_explicacao_dias.csv", index_label="data")
if len(sensibilidade):
    sensibilidade.round(3).to_csv(OUT / "xai_sensibilidade_temperatura.csv")
efeito_semana.round(2).rename("efeito").to_csv(OUT / "xai_prophet_dia_semana.csv", index_label="dia")
efeito_fer.round(2).to_csv(OUT / "xai_prophet_feriados.csv", index_label="data")

parametros = {"id_execucao": ID_EXECUCAO, "data_execucao": DATA_EXECUCAO.isoformat(timespec="seconds"),
              "versao_dados": VERSAO_DADOS, "versao_pipeline": VERSAO_PIPELINE, "ambiente": AMBIENTE,
              "horizonte_dias": HORIZONTE_DIAS, "semente": SEMENTE, "janelas": JANELAS,
              "percentil_pico": PERCENTIL_PICO, "tolerancia_empate": TOLERANCIA_EMPATE,
              "grupos_variaveis": GRUPOS, "run_escolhida": RUN_ESCOLHIDA,
              "plano": [{"run": p[0], "descricao": p[1], "modelo": p[2], "parametros": p[3], "grupos": p[4],
                         "alvo": p[5], "excluir_treino": list(p[6]), "retreino": p[7] if len(p) > 7 else None}
                        for p in PLANO]}
(OUT / "parametros_execucao_modelagem.json").write_text(json.dumps(parametros, indent=2, ensure_ascii=False))
tabela_runs

In [ ]:
r = reg.set_index("run")
def fmt(v, casas=0):
    return f"{v:,.{casas}f}".replace(",", "X").replace(".", ",").replace("X", ".")
def run_txt(k, col="MAE_validacao"):
    return f"{fmt(r.loc[k, col])} MWmed"
def ganho(nova, base, col="MAE_validacao"):
    return (1 - r.loc[nova, col] / r.loc[base, col]) * 100
media_teste = real_teste.mean()
texto_xai = []
for data, lin in explicacao.iterrows():
    partes_txt = ", ".join(f"{c} {fmt(lin[c])}" for c in explicacao.columns
                           if c not in ("exemplo", "temperatura_c", "feriado", "dia típico", "previsto", "real")
                           and abs(lin[c]) >= 1)
    texto_xai.append(f"- {lin['exemplo'].capitalize()} ({data:%d/%m/%Y}, {fmt(lin['temperatura_c'], 1)} °C). Dia típico de {fmt(lin['dia típico'])}, "
                     f"com {partes_txt}. Previsto {fmt(lin['previsto'])} e real {fmt(lin['real'])} MWmed.")
if len(sensibilidade):
    for (faixa, tipo), lin in sensibilidade.iterrows():
        if tipo == "dia útil":
            texto_xai.append(f"- Mais 1 °C em dia útil {faixa} muda a previsão em {fmt(lin['mwmed_por_grau'])} MWmed ({fmt(lin['pct_por_grau'], 2)}%).")
texto_xai.append(f"- No Prophet o domingo fica {fmt(efeito_semana['dom'], 1)}% e o sábado {fmt(efeito_semana['sáb'], 1)}% em relação à tendência, "
                 f"e o feriado mais forte de 2025 é {efeito_fer.sort_values('efeito').iloc[0]['feriado']} ({fmt(efeito_fer['efeito'].min(), 1)}%).")
texto_xai = "\n".join(texto_xai)
melhor_teste = reg[~reg["run"].isin(["R01", "FINAL"])].sort_values("MAE_teste")["run"].iloc[0]

resumo = f"""# Resumo da modelagem (Etapa 3)

Execução {ID_EXECUCAO} com os dados {ROTULO_DADOS} ({VERSAO_DADOS['linhas']} dias, de {VERSAO_DADOS['inicio']} a {VERSAO_DADOS['fim']}) e o pipeline {VERSAO_PIPELINE}.

## Problema
Prever a carga diária do subsistema Sul (MWmed) com {HORIZONTE_DIAS} dias de antecedência.
Treino de {JANELAS['treino'][0]} a {JANELAS['treino'][1]}, validação em {JANELAS['validacao'][0][:4]} para a escolha e teste em {JANELAS['teste'][0][:4]} para a confirmação.

## Resultado
- Modelo escolhido {RUN_ESCOLHIDA} ({r.loc[RUN_ESCOLHIDA, 'descricao']}).
- No teste de 2025 o modelo final, treinado com 2017 a 2024, teve MAE de {fmt(final['MAE_teste'])} MWmed, MAPE de {fmt(final['MAPE_teste'], 2)}%, RMSE de {fmt(final['RMSE_teste'])} MWmed e viés de {fmt(final['vies_teste'])} MWmed.
- A referência (repetir a semana anterior) teve MAE de {fmt(ref['MAE_teste'])} MWmed e MAPE de {fmt(ref['MAPE_teste'], 2)}% no teste. O modelo erra {fmt(final['ganho_vs_ref_teste_pct'], 1)}% menos.
- Nos dias de pico (10% maiores cargas de 2025) o MAE foi de {fmt(final['MAE_pico_teste'])} MWmed, contra {fmt(ref['MAE_pico_teste'])} da referência.
- O modelo subestimou a carga em mais de 5% em {subestimou_5} de {len(aval)} dias.
- A carga média de 2025 foi de {fmt(media_teste)} MWmed, então 1% de erro equivale a cerca de {fmt(media_teste / 100)} MWmed.
- O menor MAE no teste entre as runs foi da {melhor_teste} ({r.loc[melhor_teste, 'descricao']}), com {fmt(r.loc[melhor_teste, 'MAE_teste'])} MWmed. A escolha não muda depois de olhar o teste, porque isso seria ajustar o modelo ao teste, e a diferença fica registrada para reavaliação.

## O que cada comparação mostrou (MAE de validação 2024)
- Com temperatura a Regressão Linear foi de {run_txt('R02')} para {run_txt('R03')} ({fmt(ganho('R03', 'R02'), 1)}% de melhora). No Random Forest foi de {run_txt('R08')} para {run_txt('R07')} ({fmt(ganho('R07', 'R08'), 1)}%).
- No Random Forest o alvo absoluto teve {run_txt('R06')} e o relativo {run_txt('R07')} ({fmt(ganho('R07', 'R06'), 1)}% de melhora). O viés do absoluto na validação foi de {fmt(r.loc['R06', 'vies_validacao'])} MWmed, porque a árvore não prevê acima do que viu no treino.
- O indicador de regime na versão relativa foi um erro de desenho. O viés no teste foi de {fmt(r.loc['R04', 'vies_teste'])} MWmed com o indicador (R04) e {fmt(r.loc['R05', 'vies_teste'])} MWmed sem ele (R05), e o MAE de teste foi de {fmt(r.loc['R04', 'MAE_teste'])} e {fmt(r.loc['R05', 'MAE_teste'])} MWmed.
- O perfil de consumo da EPE deu {run_txt('R09')} com e {run_txt('R07')} sem ({fmt(ganho('R09', 'R07'), 1)}%).
- Tirar do treino os dias de grande interrupção deu {run_txt('R11')} contra {run_txt('R05')} ({fmt(ganho('R11', 'R05'), 1)}%). Ficaram fora do treino {resumo_exclusoes(r.loc['R11', 'excluidos_do_treino'])}. Sem também os eventos conhecidos, {run_txt('R12')}.
- O Gradient Boosting teve {run_txt('R10')} e o Random Forest com árvores rasas {run_txt('R13')}.
- O Prophet sem retreino teve {run_txt('R14')} na validação, mas {fmt(r.loc['R14', 'MAE_teste'])} MWmed no teste (viés de {fmt(r.loc['R14', 'vies_teste'])}), porque sem dados novos a tendência envelhece. Com retreino mensal foram {run_txt('R15')} na validação e {fmt(r.loc['R15', 'MAE_teste'])} MWmed no teste. A versão aditiva teve {run_txt('R16')}.

## Explicabilidade (modelo final)
{texto_xai}

## Limitações
- A temperatura usada é a observada (reanálise ERA5). Em produção viria da previsão do tempo para 7 dias, que tem erro, então o ganho da temperatura aqui é um limite superior.
- Os dias de grande interrupção dependem da cobertura da base da ANEEL, que tem lacuna em jun/2025.
- O perfil da EPE só existe até 2 meses depois da última publicação. Um modelo que dependa dele para de funcionar se a EPE atrasar.
"""
(OUT / "resumo_modelagem.md").write_text(resumo, encoding="utf-8")
print(resumo)

In [ ]:
pasta_zip = Path("etapa3_modelagem_pacote")
if pasta_zip.exists():
    shutil.rmtree(pasta_zip)
shutil.copytree(MOD, pasta_zip)
arquivo_zip = shutil.make_archive("etapa3_modelagem_resultados", "zip", pasta_zip)
shutil.rmtree(pasta_zip)
print("Gerado:", arquivo_zip, f"({Path(arquivo_zip).stat().st_size / 1e6:.1f} MB)")
try:
    from google.colab import files
    files.download(arquivo_zip)
except ImportError:
    pass